In [1]:
import pickle
import os.path as osp
import os
import numpy as np

results_dir = osp.relpath("./data/results/LOO_3e-3")

files = os.listdir(results_dir)

angle_errs = []
joint_errs = []
vertex_errs = []
jitter_errs = []

for file in files:
  fpath = osp.join(results_dir, file)

  with open(fpath, 'rb') as f:
    err_dict = pickle.load(f)
    angle_errs.append(err_dict['angle_error'])
    joint_errs.append(err_dict['joint_error'])
    vertex_errs.append(err_dict['vertex_error'])
    jitter_errs.append(err_dict['jitter'])
    
    print('file ' + file)
    print(f'angle error {np.mean(angle_errs[-1])}')
    print(f'joint error {np.mean(joint_errs[-1])}')
    print(f'vertex error {np.mean(vertex_errs[-1])}')
    print(F'jitter {np.mean(jitter_errs[-1])}')
    print()



file helen_error_metrics.pkl
angle error 0.7868732414185559
joint error 13.560954144628294
vertex error 17.68660605405323
jitter 0.0004374930663268052

file jin_error_metrics.pkl
angle error 0.9644630190689003
joint error 14.296971578694501
vertex error 20.191386480505503
jitter 0.00042015375516335393

file evan_error_metrics.pkl
angle error 1.579418330216555
joint error 19.39005584002555
vertex error 29.722245989551453
jitter 0.0006493645500277868

file vidya_error_metrics.pkl
angle error 0.9671286323055406
joint error 14.180344865339231
vertex error 21.79175397987104
jitter 0.0005547437698444664

file maggie_error_metrics.pkl
angle error 0.8668438541366303
joint error 13.735077385328621
vertex error 18.748751256546885
jitter 0.00036365625337279645

file michelle_error_metrics.pkl
angle error 0.8911734111607502
joint error 13.749907333555022
vertex error 20.056172924307795
jitter 0.0005721246569880337

file jason_error_metrics.pkl
angle error 1.030643148020095
joint error 16.019567509

In [2]:
angle_errs = np.array(angle_errs)
joint_errs = np.array(joint_errs)
vertex_errs = np.array(vertex_errs)
jitter = np.array(jitter_errs)

In [3]:
# overall statistics
print(f'average angle error {np.mean(angle_errs)}')
print(f'average joint errors {np.mean(joint_errs)}')
print(f'average vertex errors {np.mean(vertex_errs)}')
print(f'average jitter {np.mean(jitter)}')

average angle error 0.9815615779169747
average joint errors 14.668968137716144
average vertex errors 21.083108914526893
average jitter 0.0005141390751412188


In [18]:
print(f'average joint errors {np.mean(joint_errs, axis=0)}')
print(F'joint err at left ankle {np.mean(joint_errs, axis=0)[7]}')
print(F'joint err at right ankle {np.mean(joint_errs, axis=0)[8]}')

average joint errors [ 0.          4.83618915  5.59241208  2.9190638  15.8719788  15.06041384
  5.14729994 18.02508426 18.09539371  6.26106316 21.23147752 22.21635054
  9.60677133  9.89920515  8.40019044 15.13022688 18.41666265 14.78580312
 23.39729979 24.51036964 33.29626594 31.4009386 ]
joint err at left ankle 18.02508425630775
joint err at right ankle 18.095393706549356


In [20]:
import smplx
import open3d
from imu_uwb_pose.config import config

config = config()

smpl = smplx.create(config.body_model, model_type='smplx',
                            gender='neutral', use_face_contour=False,
                            batch_size=1,
                            ext='npz',
                            age='adult').to(config.device)

output = smpl()


In [44]:
verts = output.vertices.detach().cpu().numpy()
verts = verts[0]
faces = smpl.faces
average_err_verts = np.mean(vertex_errs, axis=0)

In [55]:
def show_colored_mesh(verts, vert_errs, faces, *,
                      cmap_name: str = "Reds",
                      window_name: str | None = None):
    """
    Same API as before, but with shape / value checks that surface
    the exact reason Open3D would have failed.
    """
    import numpy as np
    import open3d as o3d
    import matplotlib.cm as cm

    # --- 0. Convert & sanity-check shapes -------------------------------
    verts = np.asarray(verts,  dtype=np.float64)
    vert_errs = np.asarray(vert_errs, dtype=np.float64).reshape(-1)
    faces = np.asarray(faces, dtype=np.int32)

    assert verts.ndim == 2 and verts.shape[1] == 3, \
        f"`verts` must be (N,3); got {verts.shape}"
    assert faces.ndim == 2 and faces.shape[1] == 3, \
        f"`faces` must be (M,3); got {faces.shape}"
    assert len(verts) == len(vert_errs), \
        f"`vert_errs` ({len(vert_errs)}) != number of vertices ({len(verts)})"

    # No NaNs/Infs
    assert np.isfinite(verts).all(),  "`verts` contains NaN/Inf"
    assert np.isfinite(vert_errs).all(), "`vert_errs` contains NaN/Inf"

    # Face indices in bounds
    assert faces.min() >= 0,                "`faces` has negative indices"
    assert faces.max() < len(verts),        "`faces` index exceeds vertex count"

    # --- 1. Normalise error → [0,1] --------------------------------------
    if np.isclose(np.ptp(vert_errs), 0):
        err_norm = np.zeros_like(vert_errs)
    else:
        err_norm = (vert_errs - vert_errs.min()) / np.ptp(vert_errs)

    # --- 2. Colormap ------------------------------------------------------
    rgb = cm.get_cmap(cmap_name)(err_norm)[:, :3]

    # --- 3. Build mesh ----------------------------------------------------
    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices      = o3d.utility.Vector3dVector(verts)
    mesh.triangles     = o3d.utility.Vector3iVector(faces)
    mesh.vertex_colors = o3d.utility.Vector3dVector(rgb.astype(np.float64))
    mesh.compute_vertex_normals()

    # --- 4. Visualise -----------------------------------------------------
    if window_name is None:
        window_name = "Per-vertex error visualisation"
    o3d.visualization.draw_geometries([mesh],
                                      window_name=window_name,
                                      mesh_show_back_face=True,
                                      width=900, height=700)


In [56]:
show_colored_mesh(verts, average_err_verts, faces)

/tmp/ipykernel_272820/326036759.py:39: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rgb = cm.get_cmap(cmap_name)(err_norm)[:, :3]
